# 05. Conformal Prediction for Uncertainty-Aware Fraud Classification

This notebook applies split conformal prediction to the selected classifier. The model is trained on the training set, calibrated on the calibration set, and evaluated on the held-out test set.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.append(str(Path.cwd().parent))
from src.upi_fraud_pipeline import clean_dataframe, infer_target_column, prepare_features, split_train_calibration_test, train_and_compare_models, compute_conformal_threshold, generate_prediction_sets, evaluate_conformal_prediction

df = pd.read_csv(Path.cwd().parent / 'data' / 'processed' / 'cleaned_upi_dataset.csv')
target_col = infer_target_column(df)
df = clean_dataframe(df, target_column=target_col)
X, y, _, _ = prepare_features(df, target_col)
X_train, X_cal, X_test, y_train, y_cal, y_test = split_train_calibration_test(X, y)
results = train_and_compare_models(X_train, X_cal, X_test, y_train, y_cal, y_test)
primary_name = max(results, key=lambda name: results[name]['metrics']['F1'])
primary_model = results[primary_name]['model']
print('Selected primary model:', primary_name)

In [ ]:
for alpha in [0.10, 0.05, 0.01]:
    threshold = compute_conformal_threshold(primary_model, X_cal, y_cal, alpha=alpha)
    pred_sets = generate_prediction_sets(primary_model, X_test, threshold)
    metrics = evaluate_conformal_prediction(pred_sets, y_test, target_coverage=1.0 - alpha)
    print('alpha =', alpha, 'threshold =', round(threshold, 4))
    print(metrics)

## Conformal prediction interpretation

- `{Genuine}` means the selected conformal set contains only the genuine class.
- `{Fraud}` means only the fraud class is included.
- `{Genuine, Fraud}` means the example is uncertain and requires manual review.

The split conformal threshold is based on the nonconformity score $S(x, y) = 1 - P(y id x)$, computed on the calibration set.